# TCL Phase 1 Kaggle Runs

Runs inference v2 for one model on the three Phase 1 datasets (TriviaQA, SQuAD, NQ-Open), as fixed in `TCL-Phase1-Kaggle-Protocol.md`.

Run one session per model. Each session writes a zip to `/kaggle/working/` that I download and analyze locally.

## Settings

- Accelerator: **GPU T4 x2** (the NLI model for semantic entropy runs on the second GPU).
- Internet: **on** (downloads Qwen, Phi, and the NLI model from Hugging Face).
- For Gemma: attach the Kaggle model `google/gemma`, variant `transformers/2b-it`, version 3.

In [ ]:
# Pin the exact code version used for this run.
REPO = "https://github.com/awabmoha/truth-calibration-layer.git"
COMMIT = "REPLACE_WITH_COMMIT_SHA"

# Choose one model per session.
MODEL_TAG = "qwen"   # "qwen", "phi", or "gemma"
MODELS = {
    "qwen": "Qwen/Qwen2.5-0.5B-Instruct",
    "phi": "microsoft/Phi-3.5-mini-instruct",
    "gemma": "/kaggle/input/models/google/gemma/transformers/2b-it/3",
}
DATASETS = ["triviaqa", "squad", "nq_open"]
LIMIT = None         # set to e.g. 20 for a quick check first
print(MODEL_TAG, MODELS[MODEL_TAG], LIMIT)

In [ ]:
import os, subprocess
if not os.path.exists("truth-calibration-layer"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("truth-calibration-layer")
subprocess.run(["git", "checkout", COMMIT], check=True)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
os.chdir("tcl_experiments")

In [ ]:
!python scripts/check_runtime.py
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Gemma Path Check

Kaggle mounts attached models under `/kaggle/input/`. If the Gemma path above does not exist, this cell lists what is mounted so the path can be corrected.

In [ ]:
import glob
if MODEL_TAG == "gemma" and not os.path.exists(MODELS["gemma"]):
    print("Gemma path not found. Mounted candidates:")
    for p in glob.glob("/kaggle/input/**/config.json", recursive=True):
        print("  ", os.path.dirname(p))

## Run

Each dataset is resumable: if the session stops, re-running this cell continues where it left off.

In [ ]:
OUT_ROOT = f"/kaggle/working/phase1_kaggle/{MODEL_TAG}"
for ds in DATASETS:
    out_dir = f"{OUT_ROOT}/{ds}"
    cmd = [
        "python", "scripts/run_inference_v2.py",
        "--model", MODELS[MODEL_TAG],
        "--model-tag", MODEL_TAG,
        "--dataset", ds,
        "--questions", f"data/phase1_kaggle/{ds}.csv",
        "--out-dir", out_dir,
        "--se-splits", "test",
        "--resume",
    ]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

## Package Output

In [ ]:
import shutil, json
for ds in DATASETS:
    path = f"{OUT_ROOT}/{ds}/records.jsonl"
    if os.path.exists(path):
        with open(path) as f:
            n = sum(1 for _ in f)
        print(ds, n, "records")
zip_path = shutil.make_archive(f"/kaggle/working/phase1_kaggle_{MODEL_TAG}", "zip", OUT_ROOT)
print(zip_path, os.path.getsize(zip_path) // 1_000_000, "MB")